## Imports

In [1]:

import tensorflow as tf
import os
import sys


from preprocess import create_data_generators, get_class_map

# import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau,ModelCheckpoint

from pathlib import Path

## Constants


In [2]:
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 20

## Paths

In [3]:
from pathlib import Path
import os

PROJECT_BASE_DIR = Path(os.environ["NEURODETECT_BASE_DIR"])

TRAIN_DIR = str(PROJECT_BASE_DIR / "train")
TEST_DIR = str(PROJECT_BASE_DIR / "test")
MODELS_DIR = PROJECT_BASE_DIR / "models"

print("Project base:", PROJECT_BASE_DIR)
print("Train path:", TRAIN_DIR)
print("Test path:", TEST_DIR)
print("Models path:", MODELS_DIR)

print("Train exists:", os.path.exists(TRAIN_DIR))
print("Test exists:", os.path.exists(TEST_DIR))
print("Models exists:", os.path.exists(MODELS_DIR))

Project base: G:\My Drive\SeniorProject\datasets\dataset
Train path: G:\My Drive\SeniorProject\datasets\dataset\train
Test path: G:\My Drive\SeniorProject\datasets\dataset\test
Models path: G:\My Drive\SeniorProject\datasets\dataset\models
Train exists: True
Test exists: True
Models exists: True


## Train Gen

In [4]:
train_generator, test_generator = create_data_generators(
    TRAIN_DIR,
    TEST_DIR,
    IMG_SIZE,
    BATCH_SIZE
)

labels = get_class_map(train_generator)

Found 2870 images belonging to 4 classes.
Found 394 images belonging to 4 classes.


## Batch

In [5]:
X_batch, y_batch = next(train_generator) 
print("Batch image shape:", X_batch.shape)  

print("\nLoading Test Data:")

X_batch, y_batch = next(test_generator)  
print("Batch image shape:", X_batch.shape) 

Batch image shape: (32, 224, 224, 3)

Loading Test Data:
Batch image shape: (32, 224, 224, 3)


## Baseline Model

In [6]:
num_classes = len(labels)

model_baseline = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Conv2D(128, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.5),

    tf.keras.layers.Dense(num_classes, activation='softmax')
])

model_baseline.summary()

d:\SE\SP\SeniorProject\venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,169,476 (42.61 MB)

 Trainable params: 11,169,476 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

## Compile

In [7]:
model_baseline.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

## Run Epochs

In [8]:
import time
import math

steps_per_epoch = math.ceil(train_generator.samples / BATCH_SIZE)
validation_steps = math.ceil(test_generator.samples / BATCH_SIZE)

start = time.time()
history_baseline = model_baseline.fit(
    train_generator,
    steps_per_epoch=steps_per_epoch,
    validation_data=test_generator,
    validation_steps=validation_steps,
    epochs=EPOCHS
)
train_time_sec = time.time() - start

print("Training time (sec):", round(train_time_sec, 2))

Epoch 1/20
71/90 ━━━━━━━━━━━━━━━━━━━━ 8:24 27s/step - accuracy: 0.3575 - loss: 1.4412

KeyboardInterrupt: 

## Results

In [1]:
baseline_loss, baseline_acc = model_baseline.evaluate(
    test_generator,
    steps=validation_steps
)
print("Baseline Loss:", baseline_loss)
print("Baseline Accuracy:", baseline_acc)

NameError: name 'model_baseline' is not defined

## Analysis

In [2]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

# reset generator to start
test_generator.reset()

y_prob = model_baseline.predict(test_generator, steps=validation_steps)
y_pred = np.argmax(y_prob, axis=1)

y_true = test_generator.classes  # integer labels in directory order
class_names = list(test_generator.class_indices.keys())

cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:\n", cm)

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=class_names))

NameError: name 'test_generator' is not defined

## Save Model

In [ ]:
os.makedirs(MODELS_DIR, exist_ok=True)

baseline_model_path = MODELS_DIR / "baseline_cnn.keras"
model_baseline.save(str(baseline_model_path))
print(f"Saved to {baseline_model_path}")

# Transfer Model

#### Phase 1

In [6]:

transfer_train_datagen= ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    brightness_range=[0.85,1.15],
    fill_mode='nearest'
)
transfer_test_datagen=ImageDataGenerator(
    preprocessing_function=preprocess_input
)
train_gen_tf=transfer_train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)
test_gen_tf=transfer_test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

Found 2870 images belonging to 4 classes.
Found 394 images belonging to 4 classes.


## Model

In [7]:
base_model=MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
base_model.trainable = False

X= base_model.output
X=GlobalAveragePooling2D()(X)
X=Dense(256, activation='relu')(X)
X=Dropout(0.4)(X)
X=Dense(128,activation='relu')(X)
X=Dropout(0.3)(X)
outputs= Dense(train_gen_tf.num_classes, activation='softmax')(X)
model_transfer=Model(inputs=base_model.input, outputs=outputs)

## Compile

In [8]:
model_transfer.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_phase1=[
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_1r=1e-6),
    ModelCheckpoint(
        str(MODELS_DIR / 'hopefully_best_transfer_phase1.keras'),
        monitor='val_accuracy', save_best_only=True
    )
]

## Epochs

In [9]:
EPOCHS_TL=20
history_transfer= model_transfer.fit(
    train_gen_tf,
    validation_data=test_gen_tf,
    epochs=EPOCHS_TL,
    callbacks=callbacks_phase1
)

Epoch 1/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 1049s 12s/step - accuracy: 0.6422 - loss: 0.8844 - val_accuracy: 0.5178 - val_loss: 1.2368 - learning_rate: 0.0010
Epoch 2/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 171s 2s/step - accuracy: 0.7728 - loss: 0.5854 - val_accuracy: 0.5939 - val_loss: 1.1636 - learning_rate: 0.0010
Epoch 3/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 192s 2s/step - accuracy: 0.7923 - loss: 0.5078 - val_accuracy: 0.5914 - val_loss: 1.2260 - learning_rate: 0.0010
Epoch 4/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 283s 3s/step - accuracy: 0.8206 - loss: 0.4534 - val_accuracy: 0.6447 - val_loss: 1.0671 - learning_rate: 0.0010
Epoch 5/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 175s 2s/step - accuracy: 0.8265 - loss: 0.4466 - val_accuracy: 0.6396 - val_loss: 1.1088 - learning_rate: 0.0010
Epoch 6/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 950s 11s/step - accuracy: 0.8498 - loss: 0.4004 - val_accuracy: 0.6497 - val_loss: 1.1405 - learning_rate: 0.0010
Epoch 7/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 224s 3s/step - accuracy: 0.8624 - loss: 0.3708 - val_

## Results

In [10]:
transfer_loss, transfer_acc = model_transfer.evaluate(test_gen_tf)
print("Transfer Learning Loss:", transfer_loss)

13/13 ━━━━━━━━━━━━━━━━━━━━ 8s 589ms/step - accuracy: 0.7183 - loss: 1.3165
Transfer Learning Loss: 1.3164604902267456


## Comparision

In [11]:
print('Baseline Accuracy:', baseline_acc)
print('Transfer Learning Accuracy:', transfer_acc)

NameError: name 'baseline_acc' is not defined

In [12]:
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False
model_transfer.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
historical_finetune= model_transfer.fit(
    train_gen_tf,
    validation_data=test_gen_tf,
    epochs=10
)

Epoch 1/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 107s 1s/step - accuracy: 0.6474 - loss: 1.2103 - val_accuracy: 0.6777 - val_loss: 1.7535
Epoch 2/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 101s 1s/step - accuracy: 0.8237 - loss: 0.5067 - val_accuracy: 0.6574 - val_loss: 1.8763
Epoch 3/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 102s 1s/step - accuracy: 0.8498 - loss: 0.4172 - val_accuracy: 0.6497 - val_loss: 1.8930
Epoch 4/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 103s 1s/step - accuracy: 0.8655 - loss: 0.3530 - val_accuracy: 0.6472 - val_loss: 1.8516
Epoch 5/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 100s 1s/step - accuracy: 0.8683 - loss: 0.3462 - val_accuracy: 0.6675 - val_loss: 1.7467
Epoch 6/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 101s 1s/step - accuracy: 0.8791 - loss: 0.3338 - val_accuracy: 0.6701 - val_loss: 1.6772
Epoch 7/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 117s 1s/step - accuracy: 0.8885 - loss: 0.2940 - val_accuracy: 0.6802 - val_loss: 1.5697
Epoch 8/10
90/90 ━━━━━━━━━━━━━━━━━━━━ 99s 1s/step - accuracy: 0.8875 - loss: 0.2838 - val_accuracy: 0.6954 - va

## Save Model

In [13]:
os.makedirs(MODELS_DIR, exist_ok=True)

transfer_model_path = MODELS_DIR / "mobilenetv2_transfer.keras"
model_transfer.save(str(transfer_model_path))
print(f"Saved to {transfer_model_path}")

Saved to G:\My Drive\SeniorProject\datasets\dataset\models\mobilenetv2_transfer.keras


#### Phase 2

### Callbacks

In [14]:
callbacks_phase2 = [
    EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7),
    ModelCheckpoint(
        str(MODELS_DIR / 'best_transfer_phase2.keras'),
        monitor='val_accuracy', save_best_only=True
    )
]

### Unfreeze + Recompile

In [15]:
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model_transfer.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

### Run

In [16]:
history_finetune = model_transfer.fit(
    train_gen_tf,
    validation_data=test_gen_tf,
    epochs=20,
    callbacks=callbacks_phase2
)

Epoch 1/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 109s 1s/step - accuracy: 0.9010 - loss: 0.2575 - val_accuracy: 0.7234 - val_loss: 1.3744 - learning_rate: 1.0000e-05
Epoch 2/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 103s 1s/step - accuracy: 0.9122 - loss: 0.2325 - val_accuracy: 0.7284 - val_loss: 1.3196 - learning_rate: 1.0000e-05
Epoch 3/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 103s 1s/step - accuracy: 0.9220 - loss: 0.2312 - val_accuracy: 0.7411 - val_loss: 1.2496 - learning_rate: 1.0000e-05
Epoch 4/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 103s 1s/step - accuracy: 0.9181 - loss: 0.2209 - val_accuracy: 0.7386 - val_loss: 1.2623 - learning_rate: 1.0000e-05
Epoch 5/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 101s 1s/step - accuracy: 0.9247 - loss: 0.2164 - val_accuracy: 0.7386 - val_loss: 1.2029 - learning_rate: 1.0000e-05
Epoch 6/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 100s 1s/step - accuracy: 0.9254 - loss: 0.2019 - val_accuracy: 0.7487 - val_loss: 1.2401 - learning_rate: 1.0000e-05
Epoch 7/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 101s 1s/step - accuracy: 0.9258 

### Compare

In [17]:
test_gen_tf.reset()
finetune_loss, finetune_acc = model_transfer.evaluate(test_gen_tf)
print(f"Phase 1 Accuracy:  {transfer_acc:.4f}")
print(f"Phase 2 Accuracy:  {finetune_acc:.4f}")

13/13 ━━━━━━━━━━━━━━━━━━━━ 9s 739ms/step - accuracy: 0.7817 - loss: 1.1368
Phase 1 Accuracy:  0.7183
Phase 2 Accuracy:  0.7817


### Save

In [18]:
os.makedirs(MODELS_DIR, exist_ok=True)
transfer_model_path = MODELS_DIR / "mobilenetv2_finetuned.keras"
model_transfer.save(str(transfer_model_path))
print(f"Saved to {transfer_model_path}")

Saved to G:\My Drive\SeniorProject\datasets\dataset\models\mobilenetv2_finetuned.keras
